# 코랩 A100 · 테스트 시점 증강(TTA) — 0.9407 모델 재학습 없이 다시 풀기

리더보드 0.9407 모델을 **다시 학습하지 않고**, test 를 여러 방식으로 풀어 확률을 평균 냅니다.

| 방식 | 내용 | 노리는 것 |
|---|---|---|
| 다중 해상도 | 같은 모델로 768 · 896px 에서도 풀기 | 작은 글씨를 못 읽어 틀린 문제 |
| 보기 순서 이동 | a·b·c·d 를 한 칸씩 밀어서 풀고 원래 순서로 되돌림 | 특정 위치 보기를 고르는 버릇 |

- 원래 640px · 원래 순서 결과는 **학습 때 저장된 확률을 재사용** (다시 안 풂)
- 해상도별 속도를 먼저 재서, `BUDGET_HOURS` 안에 들어가는 조합만 **우선순위대로** 실행
- 조합 하나 끝날 때마다 드라이브에 저장 → 세션이 끊겨도 끝난 것까지는 남음
- 마지막에 제출 파일 여러 개 생성 (전부 평균 · 해상도만 · 순서만) → 둘 다 제출해서 비교

**오늘 밤 실행 설정 (그대로 모두 실행)**
- `BUDGET_HOURS: 3.3` — 남은 83유닛 · 시간당 18.84 기준. 896px → 640px 2칸 이동 → 768px → 640px 1칸 이동 순으로 들어가는 만큼 실행
- **시작 전 필수: 리소스 → 세션 관리에서 이 노트북 하나만 남기고 나머지 세션 종료** (켜진 세션이 유닛을 계속 씀)
- Pro+ 라 백그라운드 실행 가능 → 탭을 닫고 자도 계속 돌아감
- 런타임 A100 · `MyDrive/data.zip` 확인

In [ ]:
CONFIG = {
    "RUN_NAME": "동현_qwen3-vl-8b-bf16_px640_n6714_r32a64_lr0.0002_ep2_vis_s42_colab",   # 0.9407 모델
    "BASE_REPO": "unsloth/Qwen3-VL-8B-Instruct",
    "LOAD_IN_4BIT": False,            # 학습 때와 같게 (bf16)
    "BUDGET_HOURS": 3.3,              # 남은 83유닛 ÷ 시간당 18.84 ≈ 4.4h 에서 로드·재현확인 여유를 뺀 값
    # (해상도, 보기 이동 칸수) — 위에서부터 예산 안에 드는 만큼 실행. (640, 0) 은 이미 있음
    "PASSES": [(896, 0), (640, 2), (768, 0), (896, 2), (640, 1), (640, 3), (768, 2)],
    "DRIVE_DIR": "/content/drive/MyDrive/challenge",
    "DATA_ARCHIVE": "/content/drive/MyDrive/data.zip",
    "ARCHIVE_PASSWORD": None,
}
print("대상 모델:", CONFIG["RUN_NAME"])

# 1. 드라이브 · 데이터 준비

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR  = Path(CONFIG["DRIVE_DIR"]); DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = DRIVE_DIR / "output"                        # 결과는 전부 드라이브에
CKPT_DIR   = OUTPUT_DIR / "ckpt"
SUB_DIR    = OUTPUT_DIR / "submissions"
for d in (OUTPUT_DIR, CKPT_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- 데이터 압축 해제 (코랩 로컬 디스크로 : 드라이브에서 직접 읽으면 매우 느림) ---
WORK = Path("/content/challenge"); WORK.mkdir(exist_ok=True)
if not list(WORK.rglob("train.csv")):
    arc = Path(CONFIG["DATA_ARCHIVE"])
    assert arc.exists(), f"압축 파일이 없습니다: {arc}"
    if arc.suffix == ".zip":
        subprocess.run(["unzip", "-q", str(arc), "-d", str(WORK)], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py7zr"], check=True)
        import py7zr
        with py7zr.SevenZipFile(arc, "r", password=CONFIG["ARCHIVE_PASSWORD"]) as z:
            z.extractall(WORK)
DATA_DIR = next(WORK.rglob("train.csv")).parent           # 압축 안 폴더 구조와 무관하게 찾기
PROJECT_ROOT = DATA_DIR.parent
print("DATA_DIR:", DATA_DIR, "| 출력:", OUTPUT_DIR)

# 2. 설치

In [ ]:
!pip install -q unsloth unsloth_zoo

# 3. 불러오기 · A100 확인

In [ ]:
from unsloth import FastVisionModel
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
gpu = torch.cuda.get_device_name(); vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} | {vram:.0f}GB")
assert "A100" in gpu or vram > 38, "A100 이 아닙니다. 런타임 유형을 A100 으로 바꾸고 다시 연결하세요"

# 4. test 데이터 · 0.9407 모델 위치 · 기존 확률

In [ ]:
import numpy as np, pandas as pd, time
from PIL import Image
from tqdm.auto import tqdm
Image.MAX_IMAGE_PIXELS = None

test_df    = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
def image_path(rel):
    return DATA_DIR / str(rel).replace("\\", "/")
assert image_path(test_df["path"].iloc[0]).exists(), "test 이미지 경로를 찾지 못했습니다"

RUN_DIR = CKPT_DIR / CONFIG["RUN_NAME"]
found = sorted(RUN_DIR.glob("epoch*/test_probs.npy"))
assert found, f"{RUN_DIR} 안에서 test_probs.npy 를 찾지 못했습니다 (RUN_NAME 확인)"
ADAPTER_DIR = found[-1].parent
base_probs = np.load(found[-1])
assert base_probs.shape == (len(test_df), 4)
TTA_DIR = RUN_DIR / "tta"; TTA_DIR.mkdir(exist_ok=True)
print("어댑터:", ADAPTER_DIR, "| 기존 640px 확률:", base_probs.shape)

# 5. 모델 불러오기 (학습 때와 같은 bf16)

In [ ]:
try:
    model, _ = FastVisionModel.from_pretrained(str(ADAPTER_DIR), load_in_4bit=CONFIG["LOAD_IN_4BIT"], max_seq_length=2048)
except Exception as e:
    print("어댑터 폴더 직접 로드 실패 -> 기본 모델 + 어댑터 방식:", type(e).__name__)
    from peft import PeftModel
    model, _ = FastVisionModel.from_pretrained(CONFIG["BASE_REPO"], load_in_4bit=CONFIG["LOAD_IN_4BIT"], max_seq_length=2048)
    model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
FastVisionModel.for_inference(model)

# 해상도는 load_image 가 정하므로, processor 는 넓은 범위를 허용 (32 배수로 맞춘 이미지는 그대로 통과)
from transformers import AutoProcessor
try:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"], min_pixels=256 * 256, max_pixels=1024 * 1024)
except TypeError:
    processor = AutoProcessor.from_pretrained(CONFIG["BASE_REPO"])
tok = processor.tokenizer
print(f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f}GB")

# 6. 채점 함수 · 재현 확인 (95% 이상 일치해야 진행)

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)
def build_mc_prompt(question, opts):
    return (f"{question}\n(a) {opts[0]}\n(b) {opts[1]}\n(c) {opts[2]}\n(d) {opts[3]}\n\n"
            "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.")

LETTERS = list("abcd")
LETTER_IDS = []
for ch in LETTERS:
    ids = tok.encode(ch, add_special_tokens=False)
    assert len(ids) == 1, ch
    LETTER_IDS.append(ids[0])

def load_image(rel, side):
    pix = side * side
    im = Image.open(image_path(rel))
    try:
        im.draft("RGB", (side * 2, side * 2))
    except Exception:
        pass
    im = im.convert("RGB")
    w, h = im.size
    s = (pix / (w * h)) ** 0.5
    return im.resize((max(32, round(w * s / 32) * 32), max(32, round(h * s / 32) * 32)), Image.BICUBIC)

@torch.no_grad()
def predict_probs(df, side, shift, desc):
    """반환: (N, 4) — 항상 '원래' a/b/c/d 순서의 확률"""
    out = np.zeros((len(df), 4), dtype=np.float32)
    order = [(k + shift) % 4 for k in range(4)]          # 화면 k 번째 칸에 원래 몇 번 보기를 보여줄지
    for i in tqdm(range(len(df)), desc=desc, unit="it"):
        row = df.iloc[i]
        img = load_image(row["path"], side)
        orig = [row["a"], row["b"], row["c"], row["d"]]
        msgs = [
            {"role": "system", "content": SYSTEM_INSTRUCT},
            {"role": "user", "content": [{"type": "image", "image": img},
                                         {"type": "text", "text": build_mc_prompt(row["question"], [orig[j] for j in order])}]},
        ]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(**inputs).logits[:, -1, :].float()
        p = torch.softmax(logits[0, LETTER_IDS], dim=-1).cpu().numpy()
        for k, j in enumerate(order):
            out[i, j] = p[k]
    return out

# --- 재현 확인 : 불러온 모델이 학습 직후와 같은 답을 내는지 (test 100문항) ---
t0 = time.time()
chk = predict_probs(test_df.head(100), 640, 0, "재현 확인 (640px)")
sec640 = (time.time() - t0) / 100
agree = float(np.mean(chk.argmax(1) == base_probs[:100].argmax(1)))
print(f"학습 직후 답과 일치율 {agree:.0%} | 640px 문항당 {sec640:.2f}초")
assert agree >= 0.95, "불러온 모델이 학습 때와 다르게 답합니다 (RUN_NAME / LOAD_IN_4BIT 확인)"

# 7. 속도 측정 · 예산 안에서 실행할 조합 선택

In [ ]:
# 해상도별 속도 측정 -> 예산 안에서 실행할 조합 선택
speed = {640: sec640}
for side in sorted({s for s, _ in CONFIG["PASSES"]} - {640}):
    t0 = time.time(); predict_probs(test_df.head(15), side, 0, f"속도 측정 {side}px")
    speed[side] = (time.time() - t0) / 15
N = len(test_df)
plan, used = [], 0.0
for side, shift in CONFIG["PASSES"]:
    if (TTA_DIR / f"tta_px{side}_s{shift}.npy").exists():
        plan.append((side, shift, 0.0)); continue          # 이미 끝난 조합 (세션 재시작 시 건너뜀)
    h = speed[side] * N / 3600
    if used + h <= CONFIG["BUDGET_HOURS"]:
        plan.append((side, shift, h)); used += h
print("문항당 초:", {k: round(v, 2) for k, v in speed.items()})
for side, shift, h in plan:
    print(f"  실행 예정: {side}px · 보기 {shift}칸 이동 · {'이미 있음' if h == 0 else f'{h * 60:.0f}분'}")
print(f"예상 합계 {used:.1f}시간 ≈ {used * 15:.0f}유닛")
assert plan, "예산 안에 드는 조합이 없습니다. BUDGET_HOURS 를 늘리세요"

# 8. TTA 실행 (조합마다 드라이브에 저장)

In [ ]:
for side, shift, h in plan:
    f = TTA_DIR / f"tta_px{side}_s{shift}.npy"
    if f.exists():
        continue
    probs = predict_probs(test_df, side, shift, f"TTA {side}px · 이동{shift}")
    np.save(f, probs)
    agree = float(np.mean(probs.argmax(1) == base_probs.argmax(1)))
    print(f"  저장 {f.name} | 원래 640px 답과 일치율 {agree:.4f}")

# 9. 평균 · 제출 파일 생성

In [ ]:
L = np.array(LETTERS)
parts = {(640, 0): base_probs}
for f in sorted(TTA_DIR.glob("tta_px*_s*.npy")):
    side, shift = int(f.stem.split("_px")[1].split("_s")[0]), int(f.stem.split("_s")[-1])
    parts[(side, shift)] = np.load(f)
print("모인 조합:", sorted(parts))

def write(tag, keys):
    keys = [k for k in keys if k in parts]
    if len(keys) < 2:
        return
    avg = np.mean([parts[k] for k in keys], axis=0)
    ans = L[avg.argmax(1)]
    sub = sample_sub.copy()
    sub[sample_sub.columns[1]] = sub[sample_sub.columns[0]].map(dict(zip(test_df[sample_sub.columns[0]], ans)))
    assert sub[sample_sub.columns[1]].isna().sum() == 0
    path = SUB_DIR / f"submission_{tag}.csv"
    sub.to_csv(path, index=False)
    np.save(TTA_DIR / f"{tag}.npy", avg)                      # 앙상블 노트북에 넣을 확률
    changed = int(np.sum(ans != L[base_probs.argmax(1)]))
    print(f"[{tag}] 조합 {keys} | 원래와 달라진 답 {changed}개 ({changed / len(ans):.1%}) | {path.name}")

write("tta_all",  list(parts))
write("tta_res",  [k for k in parts if k[1] == 0])
write("tta_perm", [k for k in parts if k[0] == 640])
print("\n캐글에 위 파일들을 각각 제출해서 0.9407 과 비교하세요. 설명란 예: tta_all | 0.9407 모델 + TTA")